# 01 - Extract From Source
## Chinook Data Engineering Pipeline
### Purpose: Read metadata → Extract tables from Azure SQL → Pass to Raw Zone
 

## Step 1: Setup Parameters
All values must be passed as parameters — no hardcoding allowed

In [0]:
# Parameters
dbutils.widgets.text("catalog_name", "workspace")
dbutils.widgets.text("schema_name", "bronze")
dbutils.widgets.text("base_path", "/Volumes/workspace/raw_zone/chinook")
dbutils.widgets.text("table_name", "all")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
base_path = dbutils.widgets.get("base_path")
table_name = dbutils.widgets.get("table_name")

print("=" * 50)
print("PIPELINE PARAMETERS")
print("=" * 50)
print(f"Catalog     : {catalog_name}")
print(f"Schema      : {schema_name}")
print(f"Base Path   : {base_path}")
print(f"Table Name  : {table_name}")
print("=" * 50)

## Step 2: Read Active Tables From Metadata
Reading pipeline_control table to get list of tables to extract

In [0]:
# Read active tables from parent metadata table
active_tables = spark.sql(f"""
    SELECT table_name, file_name 
    FROM {catalog_name}.{schema_name}.pipeline_control 
    WHERE active_flag = 'Y'
    ORDER BY table_name
""")

print("=" * 50)
print("ACTIVE TABLES FROM METADATA")
print("=" * 50)
active_tables.show()
print(f"Total tables to process: {active_tables.count()}")

## Step 3: Extract Data From Azure SQL Source
Using Connection Manager to read each active table
No credentials hardcoded — using chinook_connection_catalog

In [0]:
from datetime import datetime

# Get list of tables
tables_list = active_tables.collect()
success_tables = []
failed_tables = []

print("=" * 50)
print("STARTING EXTRACTION")
print("=" * 50)

for row in tables_list:
    tbl = row['table_name']
    try:
        print(f"\n📥 Extracting: {tbl}")
        
        # Read from Azure SQL via Connection Manager
        df = spark.read.table(
            f"chinook_connection_catalog.chinook.{tbl}"
        )
        
        source_count = df.count()
        
        # Store as temp view for Notebook 02
        df.createOrReplaceTempView(f"temp_{tbl}")
        
        success_tables.append(tbl)
        print(f"  ✅ Success | Rows: {source_count}")
        
    except Exception as e:
        failed_tables.append(tbl)
        print(f"  ❌ Failed  | Error: {str(e)}")


## Step 4: Extraction Summary

In [0]:
print("=" * 50)
print("EXTRACTION SUMMARY")
print("=" * 50)
print(f"✅ Successful : {len(success_tables)}")
print(f"❌ Failed     : {len(failed_tables)}")
print("\nSuccessful tables:")
for t in success_tables:
    print(f"  - {t}")
if failed_tables:
    print("\nFailed tables:")
    for t in failed_tables:
        print(f"  - {t}")
print("=" * 50)
print("🎉 Notebook 01 Complete! Proceed to Notebook 02")

## Validation
Run the cell below to verify extraction was successful


In [0]:
# Validation - check one table
print("Validating Album table extraction...")
spark.sql("SELECT * FROM temp_Album LIMIT 5").show()
print("✅ Validation complete!")